# Capstone Mini Project — UrbanGrocer Business Analysis

**Role:** Data analyst at **UrbanGrocer**, an online grocery and household-goods retailer.

**Goal:** Conduct a complete, independent business analysis on the transactional database —
design and populate a relational schema, translate business questions into analytical SQL,
exclude invalid records, separate revenue from profit, detect data-quality issues, and assemble
the findings into a professional report with actionable recommendations.

Every query below is labelled with the **business question** it answers. The full written report
is in `Business_Report.md`; this notebook is the executable evidence behind it.

> All data is **synthetic sample data** created for this exercise — no real or personal information.

---
## Database design

| Table | Columns |
| --- | --- |
| `Customers`  | id, name, city, state, segment, signup_date |
| `Products`   | id, name, category, price, cost |
| `Orders`     | id, customer_id, order_date, status |
| `OrderItems` | id, order_id, product_id, quantity, unit_price |
| `Payments`   | id, order_id, mode, amount, date |

The data is deliberately seeded with: **one cancelled order**, **one inactive customer**,
**one order with a missing payment**, and **two orders with payment discrepancies** — so the
data-quality checks have something to find.

## Step 1 — Create and populate the database

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE Customers (
    CustomerID   INTEGER PRIMARY KEY,
    CustomerName TEXT, City TEXT, State TEXT, Segment TEXT, SignupDate TEXT
);
CREATE TABLE Products (
    ProductID   TEXT PRIMARY KEY,
    ProductName TEXT, Category TEXT, Price INTEGER, Cost INTEGER
);
CREATE TABLE Orders (
    OrderID    TEXT PRIMARY KEY,
    CustomerID INTEGER, OrderDate TEXT, Status TEXT
);
CREATE TABLE OrderItems (
    OrderItemID INTEGER PRIMARY KEY,
    OrderID TEXT, ProductID TEXT, Quantity INTEGER, UnitPrice INTEGER
);
CREATE TABLE Payments (
    PaymentID   TEXT PRIMARY KEY,
    OrderID TEXT, PaymentMode TEXT, PaidAmount INTEGER, PaymentDate TEXT
);
""")

print("Schema created.")

In [ ]:
# --- Customers (one of them, Manish Verma, never places a completed order) ---
customers = [
    (1,  "Aarav Sharma",  "Mumbai",    "Maharashtra", "Premium", "2024-01-15"),
    (2,  "Priya Menon",   "Kochi",     "Kerala",      "Regular", "2024-03-22"),
    (3,  "Rohan Gupta",   "Delhi",     "Delhi",       "Premium", "2023-11-08"),
    (4,  "Sara Khan",     "Mumbai",    "Maharashtra", "Regular", "2024-06-30"),
    (5,  "Aditya Rao",    "Bengaluru", "Karnataka",   "Premium", "2024-02-19"),
    (6,  "Neha Joshi",    "Pune",      "Maharashtra", "Regular", "2025-01-10"),
    (7,  "Karthik Nair",  "Bengaluru", "Karnataka",   "Premium", "2024-09-05"),
    (8,  "Isha Patel",    "Ahmedabad", "Gujarat",     "Regular", "2024-04-17"),
    (9,  "Manish Verma",  "Delhi",     "Delhi",       "Regular", "2025-02-28"),
    (10, "Lakshmi Iyer",  "Chennai",   "Tamil Nadu",  "Premium", "2023-12-12"),
]

# --- Products: 14 SKUs across 5 categories ---
products = [
    ("G1",  "Organic Bananas (1kg)",    "Produce",       80,  50),
    ("G2",  "Fresh Spinach (500g)",     "Produce",       40,  22),
    ("G3",  "Tomatoes (1kg)",           "Produce",       60,  35),
    ("G4",  "Full Cream Milk (1L)",     "Dairy & Eggs",  70,  55),
    ("G5",  "Brown Eggs (12)",          "Dairy & Eggs", 120,  85),
    ("G6",  "Greek Yogurt (400g)",      "Dairy & Eggs", 110,  70),
    ("G7",  "Cold Pressed Juice (1L)",  "Beverages",    220, 140),
    ("G8",  "Sparkling Water (6-pack)", "Beverages",    300, 180),
    ("G9",  "Masala Chai (250g)",       "Beverages",    250, 150),
    ("G10", "Dish Wash Gel (1L)",       "Household",    180, 110),
    ("G11", "Laundry Detergent (2kg)",  "Household",    450, 300),
    ("G12", "Paper Towels (6 rolls)",   "Household",    240, 150),
    ("G13", "Potato Chips (200g)",      "Snacks",        90,  50),
    ("G14", "Mixed Nuts (500g)",        "Snacks",       600, 420),
]

# --- Orders: O-2006 is Cancelled; customer 9 has no orders ---
orders = [
    ("O-2001", 1,  "2026-01-08", "Completed"),
    ("O-2002", 2,  "2026-01-15", "Completed"),
    ("O-2003", 3,  "2026-01-22", "Completed"),
    ("O-2004", 1,  "2026-02-03", "Completed"),
    ("O-2005", 5,  "2026-02-11", "Completed"),
    ("O-2006", 4,  "2026-02-20", "Cancelled"),
    ("O-2007", 6,  "2026-03-05", "Completed"),
    ("O-2008", 3,  "2026-03-14", "Completed"),
    ("O-2009", 7,  "2026-03-25", "Completed"),
    ("O-2010", 2,  "2026-04-02", "Completed"),
    ("O-2011", 8,  "2026-04-18", "Completed"),
    ("O-2012", 1,  "2026-04-27", "Completed"),
    ("O-2013", 5,  "2026-05-09", "Completed"),
    ("O-2014", 10, "2026-05-21", "Completed"),
    ("O-2015", 7,  "2026-06-04", "Completed"),
    ("O-2016", 4,  "2026-06-15", "Completed"),
]

order_items = [
    (1,  "O-2001", "G1",  2, 80), (2,  "O-2001", "G4",  3, 70), (3,  "O-2001", "G11", 1, 450),
    (4,  "O-2002", "G7",  2, 220),(5,  "O-2002", "G13", 1, 90),
    (6,  "O-2003", "G5",  1, 120),(7,  "O-2003", "G6",  2, 110),(8,  "O-2003", "G12", 1, 240),
    (9,  "O-2004", "G14", 1, 600),(10, "O-2004", "G9",  1, 250),
    (11, "O-2005", "G1",  1, 80), (12, "O-2005", "G2",  2, 40), (13, "O-2005", "G3",  1, 60),
    (14, "O-2005", "G4",  2, 70),
    (15, "O-2006", "G8",  1, 300),(16, "O-2006", "G10", 1, 180),     # Cancelled order
    (17, "O-2007", "G13", 3, 90), (18, "O-2007", "G7",  1, 220),
    (19, "O-2008", "G11", 1, 450),(20, "O-2008", "G12", 2, 240),
    (21, "O-2009", "G14", 2, 600),(22, "O-2009", "G8",  1, 300),
    (23, "O-2010", "G4",  4, 70), (24, "O-2010", "G5",  2, 120),
    (25, "O-2011", "G1",  2, 80), (26, "O-2011", "G3",  2, 60), (27, "O-2011", "G6",  1, 110),
    (28, "O-2012", "G7",  3, 220),(29, "O-2012", "G9",  2, 250),
    (30, "O-2013", "G11", 1, 450),(31, "O-2013", "G12", 1, 240),(32, "O-2013", "G10", 2, 180),
    (33, "O-2014", "G14", 1, 600),(34, "O-2014", "G6",  2, 110),(35, "O-2014", "G5",  1, 120),
    (36, "O-2015", "G8",  2, 300),(37, "O-2015", "G9",  1, 250),
    (38, "O-2016", "G1",  3, 80), (39, "O-2016", "G2",  1, 40), (40, "O-2016", "G13", 2, 90),
]

# --- Payments: O-2016 missing; O-2007 (+10) and O-2010 (-20) are discrepancies ---
payments = [
    ("PAY-1",  "O-2001", "UPI",        820,  "2026-01-08"),
    ("PAY-2",  "O-2002", "Card",       530,  "2026-01-15"),
    ("PAY-3",  "O-2003", "UPI",        580,  "2026-01-22"),
    ("PAY-4",  "O-2004", "Card",       850,  "2026-02-03"),
    ("PAY-5",  "O-2005", "NetBanking", 360,  "2026-02-11"),
    ("PAY-6",  "O-2007", "UPI",        500,  "2026-03-05"),   # overpaid by 10
    ("PAY-7",  "O-2008", "Card",       930,  "2026-03-14"),
    ("PAY-8",  "O-2009", "UPI",        1500, "2026-03-25"),
    ("PAY-9",  "O-2010", "Card",       500,  "2026-04-02"),   # underpaid by 20
    ("PAY-10", "O-2011", "UPI",        390,  "2026-04-18"),
    ("PAY-11", "O-2012", "NetBanking", 1160, "2026-04-27"),
    ("PAY-12", "O-2013", "Card",       1050, "2026-05-09"),
    ("PAY-13", "O-2014", "UPI",        940,  "2026-05-21"),
    ("PAY-14", "O-2015", "Card",       850,  "2026-06-04"),
    # O-2016 has NO payment row (data-quality issue)
]

cur.executemany("INSERT INTO Customers  VALUES (?,?,?,?,?,?)", customers)
cur.executemany("INSERT INTO Products   VALUES (?,?,?,?,?)",   products)
cur.executemany("INSERT INTO Orders     VALUES (?,?,?,?)",     orders)
cur.executemany("INSERT INTO OrderItems VALUES (?,?,?,?,?)",   order_items)
cur.executemany("INSERT INTO Payments   VALUES (?,?,?,?,?)",   payments)
conn.commit()

print("Database created and populated.")

## Step 2 — Analytical queries

Helper `run()` returns each result as a DataFrame. Throughout, **only `Completed` orders count
toward revenue** — cancelled orders are excluded by business judgement.

In [ ]:
def run(sql):
    """Execute SQL against the in-memory DB and return a DataFrame."""
    return pd.read_sql(sql, conn)

### Q1 — Baseline KPIs
Total revenue, completed orders, active customers, and average order value (completed orders only).

In [ ]:
run("""
SELECT SUM(oi.Quantity * oi.UnitPrice)                                     AS TotalRevenue,
       COUNT(DISTINCT o.OrderID)                                           AS CompletedOrders,
       COUNT(DISTINCT o.CustomerID)                                        AS ActiveCustomers,
       ROUND(SUM(oi.Quantity * oi.UnitPrice) * 1.0
             / COUNT(DISTINCT o.OrderID), 2)                              AS AvgOrderValue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
""")

### Q2 — Category analysis
Revenue, profit, and margin % per category. Profit separates *what we sell* from *what we keep*.

In [ ]:
run("""
SELECT p.Category,
       SUM(oi.Quantity * oi.UnitPrice)                AS Revenue,
       SUM(oi.Quantity * (oi.UnitPrice - p.Cost))     AS Profit,
       ROUND(100.0 * SUM(oi.Quantity * (oi.UnitPrice - p.Cost))
             / SUM(oi.Quantity * oi.UnitPrice), 1)    AS MarginPct
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.Category
ORDER BY Profit DESC
""")

### Q3a — Top five customers by revenue

In [ ]:
run("""
SELECT c.CustomerName, c.City, c.Segment,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.CustomerID
ORDER BY Revenue DESC
LIMIT 5
""")

### Q3b — Revenue by customer segment

In [ ]:
run("""
SELECT c.Segment,
       COUNT(DISTINCT o.CustomerID)    AS Customers,
       COUNT(DISTINCT o.OrderID)       AS Orders,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.Segment
ORDER BY Revenue DESC
""")

### Q3c — Inactive customers
Registered customers who have never placed a completed order.

In [ ]:
run("""
SELECT c.CustomerID, c.CustomerName, c.City, c.Segment, c.SignupDate
FROM Customers c
WHERE c.CustomerID NOT IN (
    SELECT DISTINCT CustomerID FROM Orders WHERE Status = 'Completed'
)
ORDER BY c.CustomerID
""")

### Q3d — Repeat vs. one-time customers
A customer is *Repeat* if they have 2+ completed orders, else *One-time*.

In [ ]:
run("""
SELECT CASE WHEN OrderCount >= 2 THEN 'Repeat' ELSE 'One-time' END AS CustomerType,
       COUNT(*)        AS Customers,
       SUM(Revenue)    AS Revenue
FROM (
    SELECT o.CustomerID,
           COUNT(DISTINCT o.OrderID)       AS OrderCount,
           SUM(oi.Quantity * oi.UnitPrice) AS Revenue
    FROM Orders o
    JOIN OrderItems oi ON o.OrderID = oi.OrderID
    WHERE o.Status = 'Completed'
    GROUP BY o.CustomerID
)
GROUP BY CustomerType
ORDER BY Revenue DESC
""")

### Q4 — Product analysis
Best sellers by **units** and by **revenue** (top 5 each).

In [ ]:
print("Top 5 by units sold")
display(run("""
SELECT p.ProductName, p.Category,
       SUM(oi.Quantity) AS UnitsSold
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.ProductID
ORDER BY UnitsSold DESC
LIMIT 5
"""))

print("Top 5 by revenue")
display(run("""
SELECT p.ProductName, p.Category,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.ProductID
ORDER BY Revenue DESC
LIMIT 5
"""))

### Q5 — Geographic analysis
Revenue by state and by city, with a threshold filter (`HAVING`) to surface only material markets (>= 1,000).

In [ ]:
print("Revenue by state (>= 1,000)")
display(run("""
SELECT c.State,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.State
HAVING Revenue >= 1000
ORDER BY Revenue DESC
"""))

print("Revenue by city (>= 1,000)")
display(run("""
SELECT c.City, c.State,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.City
HAVING Revenue >= 1000
ORDER BY Revenue DESC
"""))

### Q6 — Trend analysis
Monthly revenue across the analysis window.

In [ ]:
run("""
SELECT strftime('%Y-%m', o.OrderDate)  AS Month,
       COUNT(DISTINCT o.OrderID)        AS Orders,
       SUM(oi.Quantity * oi.UnitPrice)  AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
GROUP BY Month
ORDER BY Month
""")

### Q7 — Data quality
Two checks against completed orders:
1. **Missing payments** — a completed order with no payment row.
2. **Payment discrepancies** — recorded payment ≠ computed order value.

In [ ]:
print("Orders with MISSING payments")
display(run("""
SELECT o.OrderID, o.OrderDate,
       SUM(oi.Quantity * oi.UnitPrice) AS OrderValue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
LEFT JOIN Payments p ON o.OrderID = p.OrderID
WHERE o.Status = 'Completed' AND p.PaymentID IS NULL
GROUP BY o.OrderID
ORDER BY o.OrderID
"""))

print("Payment DISCREPANCIES (PaidAmount != OrderValue)")
display(run("""
SELECT o.OrderID,
       SUM(oi.Quantity * oi.UnitPrice)               AS OrderValue,
       p.PaidAmount,
       p.PaidAmount - SUM(oi.Quantity * oi.UnitPrice) AS Difference
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
JOIN Payments   p  ON o.OrderID = p.OrderID
WHERE o.Status = 'Completed'
GROUP BY o.OrderID
HAVING Difference <> 0
ORDER BY o.OrderID
"""))

## Step 3 — Visualization (SQL → Pandas → Seaborn)

Aggregate in SQL, hand the summary to Pandas, and chart it with Seaborn — the standard analytics
division of labor.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cat = run("""
SELECT p.Category,
       SUM(oi.Quantity * oi.UnitPrice)            AS Revenue,
       SUM(oi.Quantity * (oi.UnitPrice - p.Cost)) AS Profit
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.Category
ORDER BY Revenue DESC
""")

cat_m = cat.melt(id_vars="Category", value_vars=["Revenue", "Profit"],
                 var_name="Metric", value_name="Amount")

plt.figure(figsize=(8, 4.5))
sns.barplot(data=cat_m, x="Category", y="Amount", hue="Metric")
plt.title("Revenue vs. Profit by Category — UrbanGrocer")
plt.ylabel("Amount (INR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Monthly revenue trend

In [ ]:
trend = run("""
SELECT strftime('%Y-%m', o.OrderDate) AS Month,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
GROUP BY Month
ORDER BY Month
""")

plt.figure(figsize=(8, 4))
sns.lineplot(data=trend, x="Month", y="Revenue", marker="o")
plt.title("Monthly Revenue Trend — UrbanGrocer")
plt.ylabel("Revenue (INR)")
plt.tight_layout()
plt.show()

## Step 4 — Close the connection

In [ ]:
conn.close()
print("Analysis complete; connection closed.")